In [1]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated, List

import random

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.grammar.decorators import weight
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider, FullDecider, ProgressivelyTerminalDecider, PositionIndependentGrowDecider
from geneticengine.representations.tree.operators import GrowInitializer, PositionIndependentGrowInitializer, FullInitializer, RampedHalfAndHalfInitializer
from geneticengine.algorithms.gp.operators.initializers import HalfAndHalfInitializer, StandardInitializer
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.representations.grammatical_evolution.structured_ge import StructuredGrammaticalEvolutionRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker
from geneticengine.evaluation.parallel import ParallelEvaluator

from geneticengine.algorithms.gp.operators.combinators import ParallelStep, SequenceStep
from geneticengine.algorithms.gp.operators.crossover import GenericCrossoverStep
from geneticengine.algorithms.gp.operators.elitism import ElitismStep
from geneticengine.algorithms.gp.operators.mutation import GenericMutationStep
from geneticengine.algorithms.gp.operators.novelty import NoveltyStep
from geneticengine.algorithms.gp.operators.selection import LexicaseSelection, TournamentSelection

from geneticengine.solutions.individual import Individual, PhenotypicIndividual
from geneticengine.algorithms.gp.structure import GeneticStep
from geneticengine.problems import Problem
from geneticengine.random.sources import RandomSource
from geneticengine.representations.api import RepresentationWithCrossover, Representation
from geneticengine.evaluation import Evaluator
from typing import Iterator, Any, TypeVar

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_auc_score, roc_curve, auc
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

import time

import seaborn as sns
import matplotlib.pyplot as plt

import os     

from functools import lru_cache

import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
model_used = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1)
target_fpr_value = 0.05

### Functions and Data Preprocessing

In [4]:
for attemp in range(3):
    try:
        if os.path.exists('Base.csv'):
            df_orig = pd.read_csv('Base.csv')
            print("Dataset loaded successfully")
            break
        else:
            import kagglehub
            import shutil
            path = kagglehub.dataset_download("sgpjesus/bank-account-fraud-dataset-neurips-2022")
            csv_path = os.path.join(path, "Base.csv")
            shutil.copy(csv_path, "Base.csv")
            df_orig = pd.read_csv('Base.csv')
            print("Dataset downloaded and loaded successfully")
            break
    except Exception as e:
        print(f"Attempting again to download dataset due to error: {e}")
        if attemp < 2:
            time.sleep(5)
        else:
            raise e

Dataset loaded successfully


In [5]:
df_orig.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 32 columns):
 #   Column                            Non-Null Count    Dtype  
---  ------                            --------------    -----  
 0   fraud_bool                        1000000 non-null  int64  
 1   income                            1000000 non-null  float64
 2   name_email_similarity             1000000 non-null  float64
 3   prev_address_months_count         1000000 non-null  int64  
 4   current_address_months_count      1000000 non-null  int64  
 5   customer_age                      1000000 non-null  int64  
 6   days_since_request                1000000 non-null  float64
 7   intended_balcon_amount            1000000 non-null  float64
 8   payment_type                      1000000 non-null  object 
 9   zip_count_4w                      1000000 non-null  int64  
 10  velocity_6h                       1000000 non-null  float64
 11  velocity_24h                      1000

In [6]:
df_orig.head(2)

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0


In [7]:
#save the df with the values until month 6
df = df_orig
#train until month 6 and test after month 6
train_df = df[df['month'] < 5].sample(frac=1, random_state=42)
val_df = df[df['month'] == 5].sample(frac=1, random_state=42)
test_df = df[df['month'] >= 6].sample(frac=1, random_state=42)

train_val_df = pd.concat([train_df, val_df]).sample(frac=1, random_state=42)
train_val_df.drop('month', axis=1, inplace=True)
# train_df.drop('month', axis=1, inplace=True)
test_df.drop('month', axis=1, inplace=True)
val_df.drop('month', axis=1, inplace=True)

#split into X and y
X_train_val = train_val_df.drop('fraud_bool', axis=1)
y_train_val = train_val_df['fraud_bool']
# X_train = train_df.drop('fraud_bool', axis=1)
# y_train = train_df['fraud_bool']
# X_val = val_df.drop('fraud_bool', axis=1)
# y_val = val_df['fraud_bool']
X_test = test_df.drop('fraud_bool', axis=1)
y_test = test_df['fraud_bool']
print(len(df))

1000000


In [8]:
# X = df.drop(['fraud_bool'], axis=1)
# y = df['fraud_bool']
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [9]:
categorical_features = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]


encoders = {}
for feat in categorical_features:
    encoder = LabelEncoder()
    # X_train[feat] = encoder.fit_transform(X_train[feat])
    # X_val[feat] = encoder.transform(X_val[feat])
    X_train_val[feat] = encoder.fit_transform(X_train_val[feat])
    X_test[feat] = encoder.transform(X_test[feat])
    encoders[feat] = encoder

categorical_indices = [X_train_val.columns.get_loc(feat) for feat in categorical_features]

In [10]:
print(y_train_val.value_counts(),y_train_val.value_counts(), y_test.value_counts())

fraud_bool
0    786838
1      8151
Name: count, dtype: int64 fraud_bool
0    786838
1      8151
Name: count, dtype: int64 fraud_bool
0    202133
1      2878
Name: count, dtype: int64


### Baseline Model

In [11]:
feature_names = X_train_val.columns.tolist()
n_features = len(feature_names)

In [12]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

baseline_top_scores = []

for train_index, val_index in skf.split(X_train_val, y_train_val):
    X_train_fold, X_val_fold = X_train_val.iloc[train_index], X_train_val.iloc[val_index]
    y_train_fold, y_val_fold = y_train_val.iloc[train_index], y_train_val.iloc[val_index]
    
    model_baseline = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train_fold==0).sum() / (y_train_fold==1).sum())

    model_baseline.fit(X_train_fold, y_train_fold, categorical_feature=categorical_indices)

    val_probs = model_baseline.predict_proba(X_val_fold)[:,1]

    fpr, tpr, thresholds = roc_curve(y_val_fold, val_probs)

    target_fpr = target_fpr_value

    baseline_tpr_at_fpr = 0.0

    if np.any(fpr <= target_fpr):
        valid_indices = np.where(fpr <= target_fpr)[0]
        best_index = valid_indices[np.argmax(tpr[valid_indices])]
        baseline_tpr_at_fpr = tpr[best_index]

    baseline_top_scores.append(baseline_tpr_at_fpr)

baseline_tpr_at_fpr = np.min(baseline_top_scores)

print(f"Validation TPR: {baseline_tpr_at_fpr}")

Validation TPR: 0.46963562753036436


In [13]:
# df_new = df_orig.copy()

# categorical_features = [
#     "payment_type",
#     "employment_status",
#     "housing_status",
#     "source",
#     "device_os",
# ]
# for feat in categorical_features:
#     encoder = encoders[feat]
#     df_new[feat] = encoder.transform(df_new[feat])
# df_new_np = df_new.to_numpy()
# for ind in ARCHIVE_IND:
#     df_new[str(ind.get_phenotype())] = ind.get_phenotype().evaluate(df_new_np)

# df_new.columns


In [14]:
# model_baseline = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train==0).sum() / (y_train==1).sum())

# model_baseline.fit(X_train, y_train, categorical_feature=categorical_indices)

# train_probs = model_baseline.predict_proba(X_train)[:,1]

# fpr, tpr, thresholds = roc_curve(y_train, train_probs)

# target_fpr = target_fpr_value

# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     train_tpr_at_fpr = tpr[best_index]

# val_probs = model_baseline.predict_proba(X_val)[:,1]

# fpr, tpr, thresholds = roc_curve(y_val, val_probs)

# baseline_tpr_at_fpr = 0.0

# if np.any(fpr <= target_fpr):
#     valid_indices = np.where(fpr <= target_fpr)[0]
#     best_index = valid_indices[np.argmax(tpr[valid_indices])]
#     baseline_tpr_at_fpr = tpr[best_index]

# print(f"Train TPR: {train_tpr_at_fpr}, Validation TPR: {baseline_tpr_at_fpr}")

### Grammar

In [15]:
@dataclass
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

In [16]:
@weight(1.2)
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X_np):
        return X_np[:, self.index]
    
    def __str__(self):
        return feature_names[self.index]

In [17]:
#scalar -> scalar
@weight(0.2)
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) + self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

@weight(0.2)
@dataclass
class Subtract(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return (self.left.evaluate(X_np)) - (self.right.evaluate(X_np))
    
    def __str__(self):
        return f"({self.left} - {self.right})"

@weight(0.2)
@dataclass
class Multiply(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        return self.left.evaluate(X_np) * self.right.evaluate(X_np)
    
    def __str__(self):
        return f"({self.left} * {self.right})"

@weight(0.2)
@dataclass
class Divide(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X_np):
        denom = self.right.evaluate(X_np)
        denom = np.where(denom == 0, 1e-6, denom)  # Avoid division by zero
        return self.left.evaluate(X_np) / denom
    
    def __str__(self):
        return f"({self.left} / {self.right})"
    
@weight(0.2)
@dataclass
class Sqrt(Scalar):
    value: Scalar

    def evaluate(self, X_np):
        val = self.value.evaluate(X_np)
        val = np.asarray(val)
        val = np.clip(val, a_min=0.0, a_max=None)
        return np.sqrt(val)

    def __str__(self):
        return f"sqrt({self.value})"
    
@weight(0.2)
@dataclass
class Log(Scalar):
    value: Scalar

    def evaluate(self, X_np):
        val = self.value.evaluate(X_np)
        val = np.asarray(val)
        val = np.where(val <= 0, 1e-6, val)
        return np.log(val)
    
    def __str__(self):
        return f"log({self.value})"

In [18]:
grammar = extract_grammar([Add, Subtract, Multiply, Divide, Sqrt, Log, ScalarVar], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Add(right: Scalar, left: Scalar)<0.08>|
	Subtract(right: Scalar, left: Scalar)<0.08>|
	Multiply(right: Scalar, left: Scalar)<0.08>|
	Divide(right: Scalar, left: Scalar)<0.08>|
	Sqrt(value: Scalar)<0.08>|
	Log(value: Scalar)<0.08>|
	ScalarVar(index: Annotated[int])<0.50>
}


### Fitness and GP

In [19]:
ARCHIVE_TV_DF = X_train_val.copy()
# ARCHIVE_VAL_DF = X_val.copy()

ARCHIVE_TV_DF_TEMP = X_train_val.copy()

ARCHIVE_TEMP : list[Individual] = []
ARCHIVE_IND_DICT: dict[str, Individual] = {}

X_tv_np = ARCHIVE_TV_DF.to_numpy()
y_tv_np = y_train_val.to_numpy()
# X_val_np = ARCHIVE_VAL_DF.to_numpy()

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [20]:
def fitness_function(individual: Scalar): #individual -> expression
    
    if str(individual) in ARCHIVE_TV_DF.columns:
        return [0.0, 0.0, 1000.0, 1000.0]
    
    start = time.perf_counter()
    new_feature = individual.evaluate(X_tv_np)
    if new_feature.ndim == 0:
        new_feature = np.full(X_tv_np.shape[0], new_feature)

    X_augmented_full = np.c_[X_tv_np, new_feature]
    
    cv_scores = []

    for train_index, val_index in skf.split(X_tv_np, y_tv_np):
        X_train_fold, X_val_fold = X_augmented_full[train_index], X_augmented_full[val_index]
        y_train_fold, y_val_fold = y_tv_np[train_index], y_tv_np[val_index]
        
        model_fold = lgb.LGBMClassifier(n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train_fold==0).sum() / (y_train_fold==1).sum())

        model_fold.fit(X_train_fold, y_train_fold, categorical_feature=categorical_indices)
        
        val_probs = model_fold.predict_proba(X_val_fold)[:, 1]

        fpr, tpr, thresholds = roc_curve(y_val_fold, val_probs)

        target_fpr = target_fpr_value

        tpr_at_fpr = 0.0

        if np.any(fpr <= target_fpr):
            valid_indices = np.where(fpr <= target_fpr)[0]
            best_index = valid_indices[np.argmax(tpr[valid_indices])]
            tpr_at_fpr = tpr[best_index]

        cv_scores.append(tpr_at_fpr)

    min_cv_score = np.min(cv_scores)

    tpr_diff = min_cv_score - baseline_tpr_at_fpr
        
    features, num_operations = analyse_complexity(individual)

    end = time.perf_counter()
    elapsed = end - start
    return [min_cv_score, tpr_diff, num_operations, elapsed]



In [21]:
def analyse_complexity(individual: Scalar):
    if isinstance(individual, ScalarVar):
        return {individual.index}, 0 #unique feature
    
    total_features = set()
    total_operations = 1
    if hasattr(individual, 'left') and hasattr(individual, 'right'):
        left_features, left_operations = analyse_complexity(individual.left)
        right_features, right_operations = analyse_complexity(individual.right)
        total_features.update(left_features)
        total_features.update(right_features)
        total_operations += left_operations + right_operations
    elif hasattr(individual, 'arr'):
        arr_features, arr_operations = analyse_complexity(individual.arr)
        total_features.update(arr_features)
        total_operations += arr_operations
    return total_features, total_operations


In [22]:
class ArchiveStep(GeneticStep):
    def iterate(
        self,
        problem: Problem,
        evaluator: Evaluator,
        representation: Representation,
        random: RandomSource,
        population: Iterator[PhenotypicIndividual],
        target_size: int,
        generation: int,
    ) -> Iterator[PhenotypicIndividual]:
        global ARCHIVE_TEMP, baseline_tpr_at_fpr, ARCHIVE_TV_DF, X_tv_np, y_tv_np, skf, ARCHIVE_IND, categorical_indices, target_fpr_value, ARCHIVE_TV_DF_TEMP, ARCHIVE_IND_DICT
        
        for i, individual in enumerate(population):
            if individual.get_fitness(problem).fitness_components[0] > baseline_tpr_at_fpr:
                feature_new = individual.get_phenotype().evaluate(X_tv_np)
                if feature_new.ndim == 0:
                    feature_new = np.full(X_tv_np.shape[0], feature_new)
                ARCHIVE_TV_DF_TEMP[str(individual.get_phenotype())] = feature_new
                ARCHIVE_IND_DICT[str(individual.get_phenotype())] = individual
                ARCHIVE_TEMP.append(individual)
            yield individual
        
        if ARCHIVE_TEMP:
            print(f"Archive Size: {len(ARCHIVE_TEMP)}")

        # Apply RFE every 15 generations
        if ARCHIVE_TEMP and generation % 15 == 0:
            print(f"Generation {generation}: Applying RFE to archive")
            print(f"Archive Size before RFE: {ARCHIVE_TV_DF_TEMP.shape[1]} features")
            
            # Only apply RFE if we have more than 30 features
            if ARCHIVE_TV_DF_TEMP.shape[1] > 30:
                num_features_to_keep = 30
                
                model_rfe = lgb.LGBMClassifier(
                    n_estimators=50, 
                    max_depth=7, 
                    learning_rate=0.03, 
                    num_leaves=10, 
                    boosting_type='gbdt', 
                    random_state=42, 
                    n_jobs=-1, 
                    verbose=-1, 
                    scale_pos_weight=(y_tv_np==0).sum() / (y_tv_np==1).sum()
                )
                
                rfe = RFE(
                    estimator=model_rfe,
                    n_features_to_select=num_features_to_keep,
                    step=1
                )
                
                rfe.fit(ARCHIVE_TV_DF_TEMP.to_numpy(), y_tv_np)
                
                # Get selected features
                selected_mask = rfe.support_
                feature_cols = ARCHIVE_TV_DF_TEMP.columns.tolist()
                features_to_keep = [col for col, selected in zip(feature_cols, selected_mask) if selected]
                
                print(f"RFE selected {len(features_to_keep)} features")
                
                # Update BOTH ARCHIVE_TV_DF_TEMP and ARCHIVE_TV_DF
                ARCHIVE_TV_DF_TEMP = ARCHIVE_TV_DF_TEMP[features_to_keep].copy()
                ARCHIVE_TV_DF = ARCHIVE_TV_DF_TEMP.copy()
                
                # Update X_tv_np to match the new feature set
                X_tv_np = ARCHIVE_TV_DF.to_numpy()
                
                # Recompute categorical indices after column change
                categorical_indices = [ARCHIVE_TV_DF_TEMP.columns.get_loc(feat) for feat in categorical_features if feat in ARCHIVE_TV_DF_TEMP.columns]
                
                # Update individual dictionaries
                original_cols = set(X_train_val.columns)
                ARCHIVE_IND = [ARCHIVE_IND_DICT[col] for col in features_to_keep 
                              if col not in original_cols and col in ARCHIVE_IND_DICT]
                ARCHIVE_IND_DICT = {col: ARCHIVE_IND_DICT[col] for col in features_to_keep 
                                   if col in ARCHIVE_IND_DICT}

                # Recalculate baseline with selected features
                fold_baseline_scores = []
                for train_idx, val_idx in skf.split(X_tv_np, y_tv_np):
                    X_fold_train = X_tv_np[train_idx]
                    y_fold_train = y_tv_np[train_idx]
                    X_fold_val = X_tv_np[val_idx]
                    y_fold_val = y_tv_np[val_idx]

                    model = lgb.LGBMClassifier(
                        n_estimators=50, max_depth=7, learning_rate=0.03, num_leaves=10, 
                        boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, 
                        scale_pos_weight=(y_fold_train==0).sum() / (y_fold_train==1).sum()
                    )
                    model.fit(X_fold_train, y_fold_train, categorical_feature=categorical_indices)
                    
                    probs = model.predict_proba(X_fold_val)[:, 1]
                    fpr, tpr, thresholds = roc_curve(y_fold_val, probs)
                    
                    tpr_at_fpr = 0.0
                    target_fpr = target_fpr_value
                    if np.any(fpr <= target_fpr):
                        valid_indices = np.where(fpr <= target_fpr)[0]
                        best_index = valid_indices[np.argmax(tpr[valid_indices])]
                        tpr_at_fpr = tpr[best_index]
                    
                    fold_baseline_scores.append(tpr_at_fpr)
                
                baseline_tpr_at_fpr = np.min(fold_baseline_scores)
                
                print(f"New Baseline CV TPR: {baseline_tpr_at_fpr}")
                print(f"Archive shape after RFE: {ARCHIVE_TV_DF.shape}")
            else:
                print(f"Skipping RFE - only {ARCHIVE_TV_DF_TEMP.shape[1]} features (need >30)")
            
            ARCHIVE_TEMP = []

In [23]:
def lexicase_step():
    return SequenceStep(
        ArchiveStep(),
        ParallelStep(
            [
                ElitismStep(),
                # NoveltyStep(),
                SequenceStep(
                    LexicaseSelection(epsilon=True),
                    # TournamentSelection(tournament_size=3),
                    GenericCrossoverStep(0.9),
                    GenericMutationStep(0.1),
                )
            ],
            # weights=[0.05, 0.05, 0.9]
            weights=[0.1, 0.9]
        ),
    )

prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[False, False, True, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(1800),
    population_size=50,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 5)),
    random=r,
    step=lexicase_step(),
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='output.csv', 
            problem=prob, 
            fields={
                    "Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[3],
                    "TPR Test": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "TPR Test Diff": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    "Num Operations": lambda t,i,p: i.get_fitness(p).fitness_components[2],
                    'Generation': lambda t,i,p: i.metadata["generation"]
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

Archive Size: 7
Archive Size: 20
Archive Size: 28
Archive Size: 37
Archive Size: 50
Archive Size: 65
Archive Size: 78
Archive Size: 89
Archive Size: 101
Archive Size: 110
Archive Size: 123
Archive Size: 135
Archive Size: 147
Archive Size: 156
Archive Size: 167
Generation 15: Applying RFE to archive
Archive Size before RFE: 113 features
RFE selected 30 features
New Baseline CV TPR: 0.4670592565329407
Archive shape after RFE: (794989, 30)
Archive Size: 25
Archive Size: 44
Archive Size: 62
Archive Size: 78
Archive Size: 97


In [32]:
model = lgb.LGBMClassifier(n_estimators=350, max_depth=14, learning_rate=0.03, num_leaves=17, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train_val==0).sum() / (y_train_val==1).sum())
model.fit(X_train_val, y_train_val, categorical_feature=categorical_indices)
original_probs = model.predict_proba(X_test)[:,1]
fpr, tpr, thresholds = roc_curve(y_test, original_probs)
target_fpr = target_fpr_value
original_tpr_at_fpr = 0.0 
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    original_tpr_at_fpr = tpr[best_index]
print(f"Original Test TPR at FPR {target_fpr}: {original_tpr_at_fpr}")

print("Finalizing feature archive...")
original_cols = set(X_train_val.columns)
final_feature_cols = ARCHIVE_TV_DF.columns

ARCHIVE_IND = [ARCHIVE_IND_DICT[col] for col in final_feature_cols 
               if col not in original_cols and col in ARCHIVE_IND_DICT]
               
print(f"Final ARCHIVE_IND contains {len(ARCHIVE_IND)} new features.")

X_test_enhanced = X_test.copy()
X_test_np = X_test.to_numpy()

for ind in ARCHIVE_IND: 
    pheno_str = str(ind.get_phenotype())
    if pheno_str not in X_test_enhanced.columns:
        test_feature = ind.get_phenotype().evaluate(X_test_np)
        if test_feature.ndim == 0:
            test_feature = np.full(X_test_np.shape[0], test_feature)
        X_test_enhanced[pheno_str] = test_feature

X_test_enhanced = X_test_enhanced.loc[:, ~X_test_enhanced.columns.duplicated()]

model_aug = lgb.LGBMClassifier(n_estimators=350, max_depth=14, learning_rate=0.03, num_leaves=17, boosting_type='gbdt', random_state=42, n_jobs=-1, verbose=-1, scale_pos_weight= (y_train_val==0).sum() / (y_train_val==1).sum())

model_aug.fit(ARCHIVE_TV_DF, y_train_val, categorical_feature=categorical_indices) 

final_test_cols = ARCHIVE_TV_DF.columns
augmented_probs = model_aug.predict_proba(X_test_enhanced[final_test_cols])[:,1] 

fpr, tpr, thresholds = roc_curve(y_test, augmented_probs)
target_fpr = target_fpr_value
augmented_tpr_at_fpr = 0.0 
if np.any(fpr <= target_fpr):
    valid_indices = np.where(fpr <= target_fpr)[0]
    best_index = valid_indices[np.argmax(tpr[valid_indices])]
    augmented_tpr_at_fpr = tpr[best_index]
    
print(f"Augmented Test TPR at FPR {target_fpr}: {augmented_tpr_at_fpr}")
print(f"Improvement in TPR at FPR {target_fpr}: {augmented_tpr_at_fpr - original_tpr_at_fpr}")

Original Test TPR at FPR 0.05: 0.49409312022237667
Finalizing feature archive...
Final ARCHIVE_IND contains 24 new features.
Augmented Test TPR at FPR 0.05: 0.5288394718554552
Improvement in TPR at FPR 0.05: 0.03474635163307854


In [36]:
print(ARCHIVE_TV_DF.columns)

Index(['name_email_similarity', 'current_address_months_count',
       'date_of_birth_distinct_emails_4w', 'housing_status', 'device_os',
       'device_distinct_emails_8w',
       '(log(log((housing_status + customer_age))) + sqrt(((housing_status - bank_branch_count_8w) - zip_count_4w)))',
       'sqrt((email_is_free + device_distinct_emails_8w))',
       '(velocity_4w - prev_address_months_count)',
       '((keep_alive_session * prev_address_months_count) + bank_months_count)',
       'log((phone_home_valid / employment_status))',
       '(prev_address_months_count + (((employment_status + customer_age) / bank_branch_count_8w) * ((device_fraud_count * employment_status) - log(current_address_months_count))))',
       '(credit_risk_score * income)', '(income - employment_status)',
       '((name_email_similarity * ((housing_status * zip_count_4w) / customer_age)) + sqrt(((device_fraud_count * source) * sqrt(credit_risk_score))))',
       '((velocity_6h / (keep_alive_session / days_si

In [25]:
# df_new = df_orig.copy()

# categorical_features = [
#     "payment_type",
#     "employment_status",
#     "housing_status",
#     "source",
#     "device_os",
# ]
# for feat in categorical_features:
#     encoder = encoders[feat]
#     df_new[feat] = encoder.transform(df_new[feat])
# df_new_np = df_new.to_numpy()
# for ind in ARCHIVE_IND:
#     df_new[str(ind.get_phenotype())] = ind.get_phenotype().evaluate(df_new_np)

# df_new.columns


In [26]:
# df_new.to_csv('base_enhanced.csv', index=False)